# Local Training Notebook - DDoS Detection Models

This notebook trains MLPv2, CNN_LSTM, LSTM, and CNN1D models locally (no Federated Learning).

**Models:**
- MLPv2: Multi-Layer Perceptron
- CNN_LSTM: Hybrid CNN-LSTM
- LSTM: Long Short-Term Memory
- CNN1D: 1D Convolutional Neural Network


## 1. Install Dependencies


In [ ]:
# Install required packages
%pip install -q tensorflow pandas numpy scikit-learn matplotlib seaborn


## 2. Import Libraries


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Conv1D, MaxPooling1D, Flatten, LSTM
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


## 3. Data Preprocessing Functions


In [ ]:
def process_col(df):
    """Clean and process columns"""
    drop_cols = ['dt', 'src', 'dst', 'switch']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    num_cols = df.select_dtypes(include=['number']).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    obj_cols = df.select_dtypes(include=['object']).columns
    if len(obj_cols) > 0:
        df[obj_cols] = df[obj_cols].fillna(df[obj_cols].mode().iloc[0])

    if 'dur' in df.columns and 'dur_nsec' in df.columns:
        df['duration_sec'] = df['dur'] + df['dur_nsec'] / 1e9
    elif 'dur' in df.columns:
        df['duration_sec'] = df['dur']
    else:
        df['duration_sec'] = 0

    num_cols = [
        'pktcount', 'bytecount', 'duration_sec', 'flows', 'packetins',
        'pktperflow', 'byteperflow', 'pktrate', 'tx_bytes', 'rx_bytes',
        'tx_kbps', 'rx_kbps', 'tot_kbps', 'port_no'
    ]
    num_cols = [c for c in num_cols if c in df.columns]
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    return df


def normalization(df):
    """Normalize and engineer features"""
    df['duration_sec'] = pd.to_numeric(df.get('duration_sec', 0), errors='coerce').fillna(0)
    if 'port_no' in df.columns:
        df['port_no'] = pd.to_numeric(df['port_no'], errors='coerce')

    # Remove invalids
    for col in ['pktcount', 'bytecount', 'dur']:
        if col in df.columns:
            df = df[df[col] >= 0]

    # Encode categorical
    if 'protocol' in df.columns:
        le = LabelEncoder()
        df['protocol'] = le.fit_transform(df['protocol'])

    # Feature engineering
    df['pkt_per_sec'] = df['pktcount'] / (df['dur'] + 1e-5)
    df['byte_per_pkt'] = df['bytecount'] / (df['pktcount'] + 1e-5)
    df['rx_tx_ratio'] = (df['rx_bytes'] + 1) / (df['tx_bytes'] + 1)
    df['byte_rate'] = df['bytecount'] / (df['dur'] + 1e-5)

    # Log-scale skewed features
    for col in ['pktcount', 'bytecount', 'tx_bytes', 'rx_bytes', 'tot_kbps']:
        if col in df.columns:
            df[col] = np.log1p(df[col])

    # Outlier capping
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        df[col] = np.clip(df[col], df[col].quantile(0.01), df[col].quantile(0.99))

    df = df.dropna().reset_index(drop=True)
    return df


def split_dataset(df):
    """Split dataset into train and test"""
    X = df.drop(columns=['label'], errors='ignore')
    y = df['label']
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


## 4. Load and Preprocess Data


In [ ]:
# Upload dataset_sdn.csv to Colab or provide path
# For Colab: Use files.upload() or mount Google Drive

# Option 1: Upload file directly (uncomment below)
from google.colab import files
uploaded = files.upload()

# Option 2: If using Google Drive (uncomment below)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/dataset_sdn.csv')

# Load dataset
df = pd.read_csv('dataset_sdn.csv')
df.columns = df.columns.str.strip().str.lower()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts())


In [ ]:
# Preprocess data
df = process_col(df)
df = normalization(df)

# Split dataset
X_train, X_test, y_train, y_test = split_dataset(df)

# Convert to numpy arrays
X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

# Get number of features and classes
num_features = X_train.shape[1]
num_classes = len(np.unique(y_train))

# Convert labels to categorical
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

# Prepare 3D data for CNN/LSTM models
X_train_3d = np.expand_dims(X_train, axis=2)
X_test_3d = np.expand_dims(X_test, axis=2)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {num_features}")
print(f"Classes: {num_classes}")
print(f"\nTraining label distribution:")
print(f"Class 0: {np.sum(y_train == 0)} ({np.sum(y_train == 0)/len(y_train)*100:.2f}%)")
print(f"Class 1: {np.sum(y_train == 1)} ({np.sum(y_train == 1)/len(y_train)*100:.2f}%)")


In [ ]:
# Visualize raw data statistics
print("="*60)
print("RAW DATA STATISTICS")
print("="*60)
print(f"\nDataset Shape: {df.shape}")
print(f"\nColumn Types:")
print(df.dtypes.value_counts())
print(f"\nMissing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})
print(missing_df[missing_df['Missing Count'] > 0])

# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
class_counts = df['label'].value_counts().sort_index()
axes[0].bar(class_counts.index, class_counts.values, color=['#4caf50', '#f44336'], alpha=0.7)
axes[0].set_xlabel('Class Label', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Class Distribution (Bar Chart)', fontsize=14, fontweight='bold')
axes[0].set_xticks(class_counts.index)
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v, str(v), ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Pie chart
axes[1].pie(class_counts.values, labels=[f'Class {i}' for i in class_counts.index], 
            autopct='%1.1f%%', startangle=90, colors=['#4caf50', '#f44336'])
axes[1].set_title('Class Distribution (Pie Chart)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution_raw.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nClass Distribution:")
for label, count in class_counts.items():
    print(f"  Class {label}: {count} ({count/len(df)*100:.2f}%)")


In [ ]:
# Visualize feature distributions (sample of numeric features)
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
if 'label' in numeric_cols:
    numeric_cols.remove('label')

# Select top 12 features for visualization
top_features = numeric_cols[:12] if len(numeric_cols) >= 12 else numeric_cols

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(top_features):
    if idx < len(axes):
        df[col].hist(bins=50, ax=axes[idx], color='skyblue', edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')
        axes[idx].set_xlabel('Value', fontsize=9)
        axes[idx].set_ylabel('Frequency', fontsize=9)
        axes[idx].grid(True, alpha=0.3)

plt.suptitle('Feature Distributions (Raw Data)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions_raw.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nVisualized {len(top_features)} features out of {len(numeric_cols)} total numeric features")


### 4.2. Data Preprocessing Explanation

**Các bước xử lý dữ liệu:**

1. **process_col()**: 
   - Xóa các cột không cần thiết (dt, src, dst, switch)
   - Điền missing values cho numeric columns bằng median
   - Điền missing values cho object columns bằng mode
   - Tính toán duration_sec từ dur và dur_nsec
   - Chuyển đổi các cột numeric sang đúng kiểu dữ liệu

2. **normalization()**:
   - Loại bỏ các giá trị không hợp lệ (negative values)
   - Encode categorical features (protocol) bằng LabelEncoder
   - Feature engineering: tạo các features mới (pkt_per_sec, byte_per_pkt, rx_tx_ratio, byte_rate)
   - Log-scale transformation cho các features bị lệch (skewed)
   - Outlier capping: giới hạn giá trị trong khoảng 1%-99% percentile
   - Xóa các rows còn missing values sau khi xử lý


In [ ]:
# Preprocess data
print("="*60)
print("DATA PREPROCESSING")
print("="*60)

print("\nStep 1: Processing columns...")
df_processed = process_col(df.copy())
print(f"  ✓ Removed unnecessary columns")
print(f"  ✓ Filled missing values")
print(f"  ✓ Converted data types")

print("\nStep 2: Normalization and feature engineering...")
df_normalized = normalization(df_processed.copy())
print(f"  ✓ Removed invalid values")
print(f"  ✓ Encoded categorical features")
print(f"  ✓ Created new features (pkt_per_sec, byte_per_pkt, rx_tx_ratio, byte_rate)")
print(f"  ✓ Applied log-scale transformation")
print(f"  ✓ Capped outliers")

print(f"\nData shape before preprocessing: {df.shape}")
print(f"Data shape after preprocessing: {df_normalized.shape}")
print(f"Features removed/added: {df.shape[1] - df_normalized.shape[1]}")

# Show new features created
new_features = ['pkt_per_sec', 'byte_per_pkt', 'rx_tx_ratio', 'byte_rate']
created_features = [f for f in new_features if f in df_normalized.columns]
print(f"\nNew features created: {created_features}")

# Update df for next steps
df = df_normalized


### 4.3. Visualize Processed Data


In [ ]:
# Visualize processed data distributions
numeric_cols_processed = df.select_dtypes(include=['number']).columns.tolist()
if 'label' in numeric_cols_processed:
    numeric_cols_processed.remove('label')

# Select top 12 features for visualization
top_features_processed = numeric_cols_processed[:12] if len(numeric_cols_processed) >= 12 else numeric_cols_processed

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(top_features_processed):
    if idx < len(axes):
        df[col].hist(bins=50, ax=axes[idx], color='lightgreen', edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{col}', fontsize=10, fontweight='bold')
        axes[idx].set_xlabel('Value', fontsize=9)
        axes[idx].set_ylabel('Frequency', fontsize=9)
        axes[idx].grid(True, alpha=0.3)

plt.suptitle('Feature Distributions (After Preprocessing)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions_processed.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Correlation matrix heatmap
numeric_cols_corr = df.select_dtypes(include=['number']).columns.tolist()
if 'label' in numeric_cols_corr:
    numeric_cols_corr.remove('label')

# Select top 15 features for correlation matrix (to avoid overcrowding)
top_features_corr = numeric_cols_corr[:15] if len(numeric_cols_corr) >= 15 else numeric_cols_corr

correlation_matrix = df[top_features_corr + ['label']].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(
    correlation_matrix,
    annot=False,
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Correlation Coefficient'},
    fmt='.2f'
)
plt.title('Feature Correlation Matrix (Top 15 Features)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Show features most correlated with label
if 'label' in correlation_matrix.columns:
    label_corr = correlation_matrix['label'].drop('label').abs().sort_values(ascending=False)
    print("\n" + "="*60)
    print("TOP 10 FEATURES MOST CORRELATED WITH LABEL")
    print("="*60)
    print(label_corr.head(10))
    print("="*60)
    
    # Visualize top correlated features
    top_10_features = label_corr.head(10)
    plt.figure(figsize=(12, 6))
    bars = plt.barh(range(len(top_10_features)), top_10_features.values, color='steelblue', alpha=0.7)
    plt.yticks(range(len(top_10_features)), top_10_features.index)
    plt.xlabel('Absolute Correlation Coefficient', fontsize=12)
    plt.title('Top 10 Features Correlated with Label (DDoS Detection)', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (idx, val) in enumerate(top_10_features.items()):
        plt.text(val + 0.01, i, f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('top_correlated_features.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*60)
    print("GIẢI THÍCH Ý NGHĨA CORRELATION")
    print("="*60)
    print("""
Correlation Coefficient (Hệ số tương quan):
- Giá trị từ -1 đến +1
- +1: Tương quan dương hoàn hảo (tăng cùng nhau)
- -1: Tương quan âm hoàn hảo (tăng/giảm ngược nhau)
- 0: Không có tương quan
- > 0.3: Tương quan mạnh
- 0.1-0.3: Tương quan trung bình
- < 0.1: Tương quan yếu

Ý nghĩa trong DDoS Detection:
- Features có correlation cao với label là những đặc trưng quan trọng nhất
- Giúp model học được patterns để phân biệt DDoS attack vs normal traffic
- Các features này sẽ có trọng số cao trong model
    """)
    
    print("\n" + "="*60)
    print("GIẢI THÍCH TỪNG FEATURE")
    print("="*60)
    
    explanations = {
        'pktcount': {
            'corr': top_10_features.get('pktcount', 0),
            'meaning': 'Số lượng packet - DDoS attacks thường tạo ra số lượng packet rất lớn',
            'importance': 'Rất cao - Đặc trưng chính để nhận diện DDoS'
        },
        'bytecount': {
            'corr': top_10_features.get('bytecount', 0),
            'meaning': 'Tổng số bytes - DDoS attacks tạo ra traffic lớn về dung lượng',
            'importance': 'Rất cao - Chỉ số quan trọng về volume của attack'
        },
        'protocol': {
            'corr': top_10_features.get('protocol', 0),
            'meaning': 'Loại giao thức mạng - Một số protocol thường được dùng cho DDoS',
            'importance': 'Cao - Giúp phân biệt loại attack'
        },
        'flows': {
            'corr': top_10_features.get('flows', 0),
            'meaning': 'Số lượng flow connections - DDoS tạo ra nhiều connections đồng thời',
            'importance': 'Cao - Đặc trưng của flood attacks'
        },
        'pktrate': {
            'corr': top_10_features.get('pktrate', 0),
            'meaning': 'Tốc độ packet - DDoS có tốc độ packet rất cao',
            'importance': 'Trung bình-Cao - Chỉ số về intensity của attack'
        },
        'pktperflow': {
            'corr': top_10_features.get('pktperflow', 0),
            'meaning': 'Số packet trên mỗi flow - Pattern khác nhau giữa attack và normal',
            'importance': 'Trung bình - Giúp phân biệt behavior'
        },
        'dur': {
            'corr': top_10_features.get('dur', 0),
            'meaning': 'Thời lượng connection - DDoS attacks có thể kéo dài hoặc ngắn',
            'importance': 'Trung bình - Pattern về timing'
        },
        'tot_dur': {
            'corr': top_10_features.get('tot_dur', 0),
            'meaning': 'Tổng thời lượng - Tương tự dur nhưng tính tổng',
            'importance': 'Trung bình - Bổ sung thông tin về duration'
        },
        'tx_bytes': {
            'corr': top_10_features.get('tx_bytes', 0),
            'meaning': 'Bytes gửi đi - DDoS có thể có pattern gửi/nhận bất thường',
            'importance': 'Thấp-Trung bình - Một phần của traffic pattern'
        },
        'rx_bytes': {
            'corr': top_10_features.get('rx_bytes', 0),
            'meaning': 'Bytes nhận về - Tương tự tx_bytes',
            'importance': 'Thấp-Trung bình - Bổ sung thông tin về traffic direction'
        }
    }
    
    for feature, info in explanations.items():
        if feature in top_10_features.index:
            print(f"\n{feature.upper()} (Correlation: {info['corr']:.4f}):")
            print(f"  Ý nghĩa: {info['meaning']}")
            print(f"  Tầm quan trọng: {info['importance']}")
    
    print("\n" + "="*60)
    print("KẾT LUẬN")
    print("="*60)
    print("""
1. pktcount và bytecount có correlation cao nhất (>0.4) - Đây là 2 features quan trọng nhất
2. Các features về volume (packet, byte, flow) có correlation cao hơn features về timing
3. Model sẽ học tốt hơn từ các features có correlation cao
4. Có thể sử dụng feature selection để chỉ giữ lại các features quan trọng nhất
5. Correlation cao không có nghĩa là causation - cần phân tích thêm về domain knowledge
    """)


In [ ]:
# Box plots to show feature distributions by class
top_features_box = numeric_cols_processed[:8] if len(numeric_cols_processed) >= 8 else numeric_cols_processed

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(top_features_box):
    if idx < len(axes):
        df.boxplot(column=col, by='label', ax=axes[idx], grid=True)
        axes[idx].set_title(f'{col}', fontsize=11, fontweight='bold')
        axes[idx].set_xlabel('Class Label', fontsize=10)
        axes[idx].set_ylabel('Value', fontsize=10)
        axes[idx].get_figure().suptitle('')  # Remove default title

plt.suptitle('Feature Distributions by Class (Box Plots)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('feature_distributions_by_class.png', dpi=300, bbox_inches='tight')
plt.show()


### 4.4. Split Dataset and Prepare for Training


## 5. Model Creation Functions


In [ ]:
def create_mlpv2_model(num_features, num_classes):
    """Create MLPv2 model"""
    model = Sequential([
        keras.Input(shape=(num_features,)),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def create_cnn1d_model(num_features, num_classes):
    """Create CNN1D model"""
    model = Sequential([
        Conv1D(64, kernel_size=3, activation='relu', input_shape=(num_features, 1)),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        Conv1D(128, kernel_size=3, activation='relu'),
        BatchNormalization(),
        MaxPooling1D(2),
        Dropout(0.3),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def create_lstm_model(num_features, num_classes):
    """Create LSTM model"""
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(num_features, 1)),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def create_cnn_lstm_model(num_features, num_classes):
    """Create CNN_LSTM hybrid model"""
    model = Sequential([
        Conv1D(64, 3, activation='relu', input_shape=(num_features, 1)),
        MaxPooling1D(2),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


## 6. Training Configuration


In [ ]:
# Training parameters
EPOCHS = 50
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

# Models to train
MODELS = {
    'MLPv2': {'create_func': create_mlpv2_model, 'use_3d': False},
    'CNN1D': {'create_func': create_cnn1d_model, 'use_3d': True},
    'LSTM': {'create_func': create_lstm_model, 'use_3d': True},
    'CNN_LSTM': {'create_func': create_cnn_lstm_model, 'use_3d': True}
}

print(f"Training configuration:")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Validation split: {VALIDATION_SPLIT}")
print(f"\nModels to train: {list(MODELS.keys())}")


## 7. Train All Models


In [ ]:
# Dictionary to store training history and models
results = {}

for model_name, model_config in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    
    # Create model
    model = model_config['create_func'](num_features, num_classes)
    
    # Print model summary
    print(f"\n{model_name} Architecture:")
    model.summary()
    
    # Prepare data
    if model_config['use_3d']:
        train_data = X_train_3d
        test_data = X_test_3d
    else:
        train_data = X_train
        test_data = X_test
    
    # Train model
    history = model.fit(
        train_data,
        y_train_cat,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        verbose=1
    )
    
    # Evaluate on test set
    test_loss, test_accuracy = model.evaluate(test_data, y_test_cat, verbose=0)
    
    # Get predictions
    y_pred_proba = model.predict(test_data, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    # Store results
    results[model_name] = {
        'model': model,
        'history': history,
        'y_test': y_test,
        'y_pred': y_pred,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }
    
    print(f"\n{model_name} Results:")
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"  Test Accuracy: {test_accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")

print(f"\n{'='*60}")
print("All models trained successfully!")
print(f"{'='*60}")


## 8. Visualize Training History (Accuracy vs Loss)


In [ ]:
# Plot training history for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (model_name, result) in enumerate(results.items()):
    history = result['history']
    
    # Plot accuracy
    axes[idx].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    axes[idx].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    axes[idx].set_xlabel('Epoch', fontsize=12)
    axes[idx].set_ylabel('Accuracy', fontsize=12)
    axes[idx].set_title(f'{model_name} - Accuracy', fontsize=14, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Accuracy curves saved as 'accuracy_curves.png'")


In [ ]:
# Plot loss curves
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (model_name, result) in enumerate(results.items()):
    history = result['history']
    
    # Plot loss
    axes[idx].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[idx].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[idx].set_xlabel('Epoch', fontsize=12)
    axes[idx].set_ylabel('Loss', fontsize=12)
    axes[idx].set_title(f'{model_name} - Loss', fontsize=14, fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print("Loss curves saved as 'loss_curves.png'")


## 9. Confusion Matrices


In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (model_name, result) in enumerate(results.items()):
    y_test = result['y_test']
    y_pred = result['y_pred']
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Calculate metrics
    accuracy = result['accuracy']
    precision = result['precision']
    recall = result['recall']
    f1 = result['f1']
    
    # Plot heatmap
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=[f'Class {i}' for i in range(num_classes)],
        yticklabels=[f'Class {i}' for i in range(num_classes)],
        cbar_kws={'label': 'Count'},
        ax=axes[idx],
        linewidths=0.5,
        linecolor='gray'
    )
    
    axes[idx].set_title(
        f'{model_name} Confusion Matrix\n'
        f'Accuracy: {accuracy*100:.2f}% | Precision: {precision*100:.2f}% | '
        f'Recall: {recall*100:.2f}% | F1: {f1*100:.2f}%',
        fontsize=12,
        fontweight='bold'
    )
    axes[idx].set_xlabel('Predicted Label', fontsize=11)
    axes[idx].set_ylabel('True Label', fontsize=11)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrices saved as 'confusion_matrices.png'")


## 10. Metrics Comparison


In [ ]:
# Create comparison table
comparison_data = []
for model_name, result in results.items():
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{result['accuracy']*100:.2f}%",
        'Precision': f"{result['precision']*100:.2f}%",
        'Recall': f"{result['recall']*100:.2f}%",
        'F1 Score': f"{result['f1']*100:.2f}%",
        'Test Loss': f"{result['test_loss']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*60)
print("Metrics Comparison")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)


In [ ]:
# Visualize metrics comparison
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    model_names = list(results.keys())
    metric_values = [results[name][metric] * 100 for name in model_names]
    
    bars = axes[idx].bar(model_names, metric_values, color=['#4caf50', '#2196f3', '#ff9800', '#f44336'])
    axes[idx].set_ylabel(f'{metric.capitalize()} (%)', fontsize=12)
    axes[idx].set_title(f'{metric.capitalize()} Comparison', fontsize=14, fontweight='bold')
    axes[idx].set_ylim([0, 100])
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                      f'{height:.2f}%',
                      ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Metrics comparison saved as 'metrics_comparison.png'")


## 11. Detailed Classification Reports


In [ ]:
# Print detailed classification reports
for model_name, result in results.items():
    print(f"\n{'='*60}")
    print(f"{model_name} - Detailed Classification Report")
    print(f"{'='*60}")
    print(classification_report(
        result['y_test'],
        result['y_pred'],
        target_names=[f'Class {i}' for i in range(num_classes)]
    ))
    print(f"{'='*60}")


## 12. Save Models (Optional)


In [ ]:
# Save all trained models
for model_name, result in results.items():
    filename = f"{model_name}_local.h5"
    result['model'].save(filename)
    print(f"Saved {model_name} model as {filename}")

print("\nAll models saved successfully!")
